# Inspecting Data from The Master Table

This file is meant for gathering descriptive statistics regarding the SD_ma_master_table.csv, which is created after running preprocess_create_master_table.m. This aim of creating this file is to provide information regarding the feasibility of assessing several RQs.

In [50]:
import pandas as pd
import numpy as np

# Paths
data_path = "/Users/Kurvi001/Documents/Serial_Dependence/Analysis/results/SD_ma_master_table.csv"
info_path = "/Users/Kurvi001/Documents/Serial_Dependence/Study_specific_information.csv"

# Load preprocessed master table
df = pd.read_csv(data_path, sep=";")

# Inspect columns
print(df.columns.tolist())
print("Shape:", df.shape)

display(df.head(5))

['delta', 'theta', 'resp', 'error', 'rt', 'obs', 'block', 'cond', 'study', 'experiment', 'stimulus', 'expnum', 'stimtype', 'studynum', 'code', 'obsid', 'bin', 'outliers_q', 'outliers_sd', 'outliers_rt', 'error_iqr', 'errorsd', 'mfree_ss_bias_factor', 'theta_cent', 'error_ori_deb', 'error_ori_deb_sd_deb', 'error_norm', 'error_iqr_norm', 'error_ori_deb_norm', 'error_ori_deb_sd_deb_norm', 'mfree_sd_bias_factor', 'codenum']
Shape: (517596, 32)


,delta,theta,resp,error,rt,obs,block,cond,study,experiment,...,mfree_ss_bias_factor,theta_cent,error_ori_deb,error_ori_deb_sd_deb,error_norm,error_iqr_norm,error_ori_deb_norm,error_ori_deb_sd_deb_norm,mfree_sd_bias_factor,codenum
0,59.0,160,173.0,13.0,2.291509,1,1,1.0,Abreo et al. (2023),Experiment 1,...,-13.0,70,12.501635,9.297788,0.675058,0.896852,1.000243,0.767995,NaN,1
1,48.0,170,171.0,1.0,2.842005,1,1,1.0,Abreo et al. (2023),Experiment 1,...,-1.0,80,11.030791,0.603372,0.034709,0.061366,0.882562,0.031169,NaN,1
2,41.0,150,163.5,13.5,4.674623,1,1,1.0,Abreo et al. (2023),Experiment 1,...,NaN,60,13.131243,4.136979,0.701739,0.931663,1.050617,0.330632,13.131243,1
3,-64.0,61,61.5,0.5,3.782852,1,1,1.0,Abreo et al. (2023),Experiment 1,...,NaN,-29,7.372858,11.904159,0.008028,0.026554,0.589895,0.988878,NaN,1
4,-82.0,158,147.0,-11.0,7.108188,1,1,1.0,Abreo et al. (2023),Experiment 1,...,11.0,68,-12.768193,-9.988836,-0.605640,-0.774119,-1.021570,-0.866489,NaN,1


In [41]:
# Create one row per study/experiment
experiment_summary = (
    df.groupby(
        ["study", "experiment", "expnum", "stimulus"],
        dropna=False
    )
    .agg(
        n_participants=("obs", "nunique"),
        n_trials=("obs", "size"),

        # Experiments where all RTs are 9's
        rt_all_9=("rt", lambda x: int((x == 9).all())),

        # Number of missing RT values
        rt_missing=("rt", lambda x: x.isna().sum()),

        # RT descriptives
        rt_mean=("rt", "mean"),
        rt_sd=("rt", "std"),
    )
    .reset_index()
)

print("Number of experiments:", len(experiment_summary))

Number of experiments: 32


In [47]:
experiment_summary = (
    df.groupby(
        ["study", "experiment", "expnum", "stimulus"],
        dropna=False
    )
    .agg(
        n_participants=("obs", "nunique"),
        n_trials=("obs", "size"),

        # RT information
        rt_all_9=("rt", lambda x: int((x == 9).all())),
        rt_missing=("rt", lambda x: x.isna().sum()),
        rt_mean=("rt", "mean"),
        rt_sd=("rt", "std"),

        # Missing errors before preprocessing
        error_missing_original=("error", lambda x: x.isna().sum()),

        # Missing values after IQR-based cleaning + orientation correction
        error_ori_deb_missing=("error_ori_deb", lambda x: x.isna().sum()),

        # Individual outlier criteria
        outliers_q=("outliers_q", "sum"),
        outliers_sd=("outliers_sd", "sum"),
        outliers_rt=("outliers_rt", "sum"),
    )
    .reset_index()
)

# Number of new NaNs introduced during preprocessing
experiment_summary["n_outliers"] = (
    experiment_summary["error_ori_deb_missing"]
    - experiment_summary["error_missing_original"]
)

# Percentage of total trials classified as outliers
experiment_summary["outlier_percentage"] = (
    experiment_summary["n_outliers"]
    / experiment_summary["n_trials"]
    * 100
)

display(experiment_summary)

print("Number of experiments:", len(experiment_summary))
print(" min outlier percentage:", min(experiment_summary["outlier_percentage"]), 
      "\n", "max outlier percentage:", max(experiment_summary["outlier_percentage"]), 
      "\n", "standard deviation:", np.std(experiment_summary["outlier_percentage"]), 
      "\n", "mean:", np.mean(experiment_summary["outlier_percentage"]))

print(" Proportion of retained total trials:", experiment_summary["n_trials"].sum() - experiment_summary["error_ori_deb_missing"].sum(), "/", experiment_summary["n_trials"].sum())

# Load study-specific information
study_info = pd.read_csv(info_path)

# Keep only the columns we want to add
study_info_subset = study_info[
    [
        "Exp_num",
        "Stimulus-type",
        "N_trials",
        "Masked",
        "Distractor",
        "Response_method",
        "Time_pressure"
    ]
].copy()

# Rename identifiers to match experiment_summary
study_info_subset = study_info_subset.rename(columns={
    "Exp_num": "expnum",
    "Stimulus-type": "stimulus",
    "N_trials": "n_trials"
})

# Merge
experiment_summary = experiment_summary.merge(
    study_info_subset,
    on=["expnum", "stimulus", "n_trials"],
    how="left",
    validate="many_to_one"
)

display(experiment_summary)

output_path = "/Users/Kurvi001/Documents/Serial_Dependence/Analysis/results/experiment_summary.csv"

experiment_summary.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

,study,experiment,expnum,stimulus,n_participants,n_trials,rt_all_9,rt_missing,rt_mean,rt_sd,error_missing_original,error_ori_deb_missing,outliers_q,outliers_sd,outliers_rt,n_outliers,outlier_percentage
0,Abreo et al. (2023),Experiment 1,1,Orientation,36,6754,0,0,2.217456,1.210272,0,419,401,53,21,419,6.203731
1,Blonde et al. (2023),Experiment 2,1,Orientation,17,2380,0,0,1.612143,2.053867,0,144,132,33,13,144,6.050420
2,Ceylan & Pascucci (2023),Experiment 2,1,Orientation,20,6000,0,0,1.844380,3.986807,0,158,148,38,12,158,2.633333
3,Ceylan et al. (2021),Experiment 1,1,Orientation,24,9600,0,0,1.632248,0.987968,0,241,231,56,12,241,2.510417
4,Ceylan et al. (2021),Experiment 2,2,Orientation,24,9600,0,0,1.443445,0.751446,0,207,205,70,2,207,2.156250
5,Chetverikov & Jehee (2023),Experiment 1,1,Motion,18,19836,1,0,9.000000,0.000000,0,1956,678,85,0,1956,9.860859
6,Cicchini et al. (2018),Experiment 2,1,Orientation,6,12300,0,0,1.898925,1.450627,0,270,265,57,5,270,2.195122
7,Fischer & Whitney (2014),Experiment 1b,1,Orientation,4,3328,1,0,9.000000,0.000000,0,129,129,37,0,129,3.876202
8,Fischer et al. (2020),Experiment 1,1,Motion,20,32640,0,0,1.433721,1.874859,0,1866,1028,196,50,1866,5.716912
9,Fischer et al. (2020),Experiment 2,2,Motion,49,79968,0,0,1.588795,2.131247,0,3972,2099,377,187,3972,4.966987


Number of experiments: 32
 min outlier percentage: 1.2134831460674158 
 max outlier percentage: 10.694014447884417 
 standard deviation: 2.2528458941404654 
 mean: 4.05968996696244
 Proportion of retained total trials: 487082 / 517596


,study,experiment,expnum,stimulus,n_participants,n_trials,rt_all_9,rt_missing,rt_mean,rt_sd,...,error_ori_deb_missing,outliers_q,outliers_sd,outliers_rt,n_outliers,outlier_percentage,Masked,Distractor,Response_method,Time_pressure
0,Abreo et al. (2023),Experiment 1,1,Orientation,36,6754,0,0,2.217456,1.210272,...,419,401,53,21,419,6.203731,1,1,Mouse,1
1,Blonde et al. (2023),Experiment 2,1,Orientation,17,2380,0,0,1.612143,2.053867,...,144,132,33,13,144,6.050420,0,1,Mouse,0
2,Ceylan & Pascucci (2023),Experiment 2,1,Orientation,20,6000,0,0,1.844380,3.986807,...,158,148,38,12,158,2.633333,0,2,Mouse,0
3,Ceylan et al. (2021),Experiment 1,1,Orientation,24,9600,0,0,1.632248,0.987968,...,241,231,56,12,241,2.510417,1,0,Mouse,0
4,Ceylan et al. (2021),Experiment 2,2,Orientation,24,9600,0,0,1.443445,0.751446,...,207,205,70,2,207,2.156250,1,0,Mouse,0
5,Chetverikov & Jehee (2023),Experiment 1,1,Motion,18,19836,1,0,9.000000,0.000000,...,1956,678,85,0,1956,9.860859,0,0,Buttons,9
6,Cicchini et al. (2018),Experiment 2,1,Orientation,6,12300,0,0,1.898925,1.450627,...,270,265,57,5,270,2.195122,1,0,Mouse,0
7,Fischer & Whitney (2014),Experiment 1b,1,Orientation,4,3328,1,0,9.000000,0.000000,...,129,129,37,0,129,3.876202,1,0,Keys,9
8,Fischer et al. (2020),Experiment 1,1,Motion,20,32640,0,0,1.433721,1.874859,...,1866,1028,196,50,1866,5.716912,1,1,Mouse,1
9,Fischer et al. (2020),Experiment 2,2,Motion,49,79968,0,0,1.588795,2.131247,...,3972,2099,377,187,3972,4.966987,1,1,Mouse,1


Saved to: /Users/Kurvi001/Documents/Serial_Dependence/Analysis/results/experiment_summary.csv


In [54]:
# Select orientation studies, excluding Abreo et al.
ori = df[
    (df["stimulus"].str.lower() == "orientation") &
    (~df["study"].str.contains("Abreo", case=False, na=False))
].copy()

# Preserve the existing row/trial order
ori["theta_prev"] = (
    ori.groupby(
        ["study", "experiment", "expnum", "cond", "obs", "block"]
    )["theta"]
    .shift(1)
)

# Raw difference between consecutive theta values
ori["theta_diff"] = ori["theta"] - ori["theta_prev"]

# Find consecutive orientations differing by more than 90 degrees
theta_over_90 = ori[ori["theta_diff"].abs() > 90].copy()

print("Number of consecutive trial pairs with |theta difference| > 90°:",
      len(theta_over_90))

display(
    theta_over_90[
        [
            "study",
            "experiment",
            "expnum",
            "cond",
            "obs",
            "block",
            "theta_prev",
            "theta",
            "theta_diff"
        ]
    ]
)

theta_over_90_summary = (
    theta_over_90
    .groupby(
        ["study", "experiment", "expnum"],
        dropna=False
    )
    .size()
    .reset_index(name="n_theta_diff_over_90")
)

display(theta_over_90_summary)

# Count all valid consecutive trial pairs per study/experiment
theta_pair_counts = (
    ori[ori["theta_prev"].notna()]
    .groupby(
        ["study", "experiment", "expnum"],
        dropna=False
    )
    .size()
    .reset_index(name="n_consecutive_pairs")
)

# Count flagged pairs
theta_over_90_summary = (
    theta_over_90
    .groupby(
        ["study", "experiment", "expnum"],
        dropna=False
    )
    .size()
    .reset_index(name="n_theta_diff_over_90")
)

# Merge summaries
theta_over_90_summary = theta_pair_counts.merge(
    theta_over_90_summary,
    on=["study", "experiment", "expnum"],
    how="left"
)

# Experiments with no flagged pairs should show 0
theta_over_90_summary["n_theta_diff_over_90"] = (
    theta_over_90_summary["n_theta_diff_over_90"]
    .fillna(0)
    .astype(int)
)

# Percentage of consecutive pairs with a raw theta difference > 90°
theta_over_90_summary["percentage_over_90"] = (
    theta_over_90_summary["n_theta_diff_over_90"]
    / theta_over_90_summary["n_consecutive_pairs"]
    * 100
)

display(theta_over_90_summary)

Number of consecutive trial pairs with |theta difference| > 90°: 63496


,study,experiment,expnum,cond,obs,block,theta_prev,theta,theta_diff
6755,Blonde et al. (2023),Experiment 2,1,1.0,1,2,7.0,118,111.0
6768,Blonde et al. (2023),Experiment 2,1,1.0,1,2,131.0,0,-131.0
6769,Blonde et al. (2023),Experiment 2,1,1.0,1,2,0.0,122,122.0
6770,Blonde et al. (2023),Experiment 2,1,1.0,1,2,122.0,28,-94.0
6783,Blonde et al. (2023),Experiment 2,1,1.0,1,2,19.0,168,149.0
...,...,...,...,...,...,...,...,...,...
517579,Samaha et al. (2019),Experiment 1,1,1.0,20,5,22.0,128,106.0
517581,Samaha et al. (2019),Experiment 1,1,1.0,20,5,165.0,58,-107.0
517586,Samaha et al. (2019),Experiment 1,1,1.0,20,5,79.0,170,91.0
517587,Samaha et al. (2019),Experiment 1,1,1.0,20,5,170.0,73,-97.0


,study,experiment,expnum,n_theta_diff_over_90
0,Blonde et al. (2023),Experiment 2,1,578
1,Ceylan & Pascucci (2023),Experiment 2,1,1406
2,Ceylan et al. (2021),Experiment 1,1,2424
3,Ceylan et al. (2021),Experiment 2,2,2445
4,Cicchini et al. (2018),Experiment 2,1,2553
5,Fischer & Whitney (2014),Experiment 1b,1,764
6,Gallagher & Benton (2022),Experiment 1,1,4058
7,Houborg et al. (2023),Experiment 1,1,720
8,Houborg et al. (2023),Experiment 2,2,731
9,Houborg et al. (2023b),Experiment 1,1,3391


,study,experiment,expnum,n_consecutive_pairs,n_theta_diff_over_90,percentage_over_90
0,Blonde et al. (2023),Experiment 2,1,2346,578,24.637681
1,Ceylan & Pascucci (2023),Experiment 2,1,5800,1406,24.241379
2,Ceylan et al. (2021),Experiment 1,1,9552,2424,25.376884
3,Ceylan et al. (2021),Experiment 2,2,9552,2445,25.596734
4,Cicchini et al. (2018),Experiment 2,1,12095,2553,21.107896
5,Fischer & Whitney (2014),Experiment 1b,1,3296,764,23.179612
6,Gallagher & Benton (2022),Experiment 1,1,16380,4058,24.774115
7,Houborg et al. (2023),Experiment 1,1,2850,720,25.263158
8,Houborg et al. (2023),Experiment 2,2,3040,731,24.046053
9,Houborg et al. (2023b),Experiment 1,1,13836,3391,24.508528
